# El Niño 2026 — What caused it, and what it does to Indonesia
**le-gis.com climate project · Notebook 01: indices, event comparison, recharge, Indonesia impact**

Framework (hypotheses tested across the project):

| # | Mechanism | Question | This notebook |
|---|-----------|----------|---------------|
| H1 | **Recharge** (Wyrtki 1975; Jin 1997) | Was heat stored in the western Pacific before onset? | WWV → Niño-3.4 lead-lag |
| H2 | **Trigger** — westerly wind bursts / MJO (Harrison & Vecchi 1997) | Did wind reversals launch Kelvin waves? | Notebook 02 (ERA5 Hovmöller) |
| H3 | **Bjerknes feedback** (Bjerknes 1969) | Do SST, winds and SOI grow together? | SOI vs Niño-3.4; Pacific SST map |
| H4 | **Warming baseline** (van Oldenborgh et al. 2021) | How much is El Niño vs global ocean warming? | ONI vs RONI |
| — | **Indonesia impact** (+ IOD, Saji et al. 1999) | Where did the 2026 dry season fail? | CHIRPS anomaly map, provinces, time series |

Runs in Google Colab. Part A needs only internet; Part B needs an Earth Engine Cloud project.

In [ ]:
!pip -q install earthengine-api geemap
import io, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .3})

def get(url):
    r = requests.get(url, timeout=60, headers={"User-Agent": "le-gis-enso/1.0"})
    r.raise_for_status()
    return r.text

## Part A — Climate indices (NOAA CPC / PSL / PMEL)

In [ ]:
# --- Niño regions, monthly (OISST-based), CPC ---
txt = get("https://www.cpc.ncep.noaa.gov/data/indices/sstoi.indices")
d = pd.read_csv(io.StringIO(txt), sep=r"\s+")
d.columns = ["yr","mon","n12","n12a","n3","n3a","n4","n4a","n34","n34a"]
d.index = pd.to_datetime(dict(year=d.yr, month=d.mon, day=1))
nino = d[["n12a","n3a","n4a","n34a"]].rename(columns={
    "n12a":"Niño-1+2","n3a":"Niño-3","n4a":"Niño-4","n34a":"Niño-3.4"})

# --- ONI and RONI (3-month running, centred month) ---
SEAS = {s:i+1 for i,s in enumerate("DJF JFM FMA MAM AMJ MJJ JJA JAS ASO SON OND NDJ".split())}
def read_oni(url):
    t = pd.read_csv(io.StringIO(get(url)), sep=r"\s+")
    t.columns = [c.upper() for c in t.columns]
    t.index = pd.to_datetime(dict(year=t.YR, month=t.SEAS.map(SEAS), day=1))
    return t["ANOM"]
oni = read_oni("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt")
try:
    roni = read_oni("https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt")
except Exception as e:
    roni = None; print("RONI not fetched:", e, "-> download from cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/")

# --- PSL-format monthly tables (SOI, DMI) ---
def read_psl(url):
    lines = get(url).splitlines()
    y0, y1 = map(int, lines[0].split()[:2]); rows = []
    for ln in lines[1:]:
        p = ln.split()
        if len(p) == 13 and p[0].isdigit() and y0 <= int(p[0]) <= y1:
            rows += [(pd.Timestamp(int(p[0]), m, 1), float(v)) for m, v in enumerate(p[1:], 1)]
    s = pd.Series(dict(rows)).sort_index()
    return s.where(s > -90)
soi = read_psl("https://psl.noaa.gov/data/correlation/soi.data")
dmi = read_psl("https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data")

# --- Warm Water Volume (PMEL TAO), equatorial Pacific 120E-80W, 5S-5N ---
def read_wwv(url="https://www.pmel.noaa.gov/tao/wwv/data/wwv.dat"):
    rows = []
    for ln in get(url).splitlines():
        p = ln.split()
        if p and p[0].isdigit() and len(p[0]) == 6:
            rows.append((pd.Timestamp(int(p[0][:4]), int(p[0][4:]), 1), float(p[1])))
    return pd.Series(dict(rows)).sort_index()
try:
    wwv = read_wwv(); print("WWV:", wwv.index.min().date(), "->", wwv.index.max().date())
except Exception as e:
    wwv = None; print("WWV not fetched:", e, "-> check pmel.noaa.gov/elnino/upper-ocean-heat-content-and-enso")

print("Latest Niño values:\n", nino.tail(4).round(2))

In [ ]:
# 2025-2026 evolution: La Niña -> El Niño
fig, ax = plt.subplots()
nino["2025":].plot(ax=ax, lw=2)
ax.axhline(0.5, c="k", ls=":", lw=1); ax.axhline(-0.5, c="k", ls=":", lw=1)
ax.set_title("Niño region SST anomalies, 2025–2026 (°C)"); ax.set_ylabel("°C")
plt.show()

### Event comparison — 2026 vs the great El Niños
Aligned January of the onset year (Y0) to June of Y+1. Note 2023 onset in raw index was inflated by background warming — see the RONI section.

In [ ]:
EVENTS = [1982, 1997, 2015, 2023, 2026]
labels = [f"{m}\nY0" if i==0 else m for i, m in enumerate(
    pd.date_range("2000-01", periods=18, freq="MS").strftime("%b"))]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=False)
for col, ax in zip(["Niño-3.4", "Niño-1+2"], axes):
    for y in EVENTS:
        s = nino[col][f"{y}-01":f"{y+1}-06"].values
        ax.plot(range(len(s)), s, lw=3 if y == 2026 else 1.6,
                c="crimson" if y == 2026 else None, label=f"{y}/{str(y+1)[2:]}")
    ax.set_xticks(range(18)); ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(f"{col} anomaly by event (°C)")
axes[0].legend()
plt.tight_layout(); plt.show()

peak = {y: nino.loc[f"{y}-06":f"{y+1}-03"].max().round(2) for y in EVENTS}
pd.DataFrame(peak).T.rename_axis("event (peak Jun–Mar)")

### H1 — Recharge: does stored heat lead the surface?
If recharge theory holds, WWV should correlate with Niño-3.4 most strongly when it **leads** by several months (literature: ~6–9).

In [ ]:
if wwv is not None:
    df = pd.concat([wwv.rename("wwv"), nino["Niño-3.4"]], axis=1).dropna()
    lags = range(0, 16)
    r = [df["Niño-3.4"].corr(df["wwv"].shift(L)) for L in lags]
    best = int(np.argmax(r))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    ax[0].bar(lags, r, color=["crimson" if L == best else "steelblue" for L in lags])
    ax[0].set(title=f"WWV leading Niño-3.4 — best lag {best} months (r={r[best]:.2f})",
              xlabel="WWV lead (months)", ylabel="correlation")
    s = df["2024":]
    ax[1].plot(s.index, s["wwv"] / s["wwv"].abs().max(), label="WWV (scaled)", lw=2)
    ax[1].plot(s.index, s["Niño-3.4"] / s["Niño-3.4"].abs().max(), label="Niño-3.4 (scaled)", lw=2)
    ax[1].set_title("2024–2026: recharge then discharge"); ax[1].legend()
    plt.tight_layout(); plt.show()

### H3 & H4 — Atmospheric coupling (SOI) and the warming baseline (ONI vs RONI)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
c = pd.concat([nino["Niño-3.4"], soi.rename("SOI")], axis=1).dropna()
ax[0].scatter(c["Niño-3.4"], c["SOI"], s=6, alpha=.4)
last = c["2026-06":]
ax[0].scatter(last["Niño-3.4"], last["SOI"], c="crimson", s=30, label="Jun–2026 onward")
ax[0].set(title=f"Bjerknes coupling: r = {c.corr().iloc[0,1]:.2f}", xlabel="Niño-3.4 (°C)", ylabel="SOI"); ax[0].legend()

oni["1980":].plot(ax=ax[1], label="ONI (raw)", c="grey")
if roni is not None:
    roni["1980":].plot(ax=ax[1], label="RONI (relative to tropical mean)", c="crimson")
    gap = (oni - roni).dropna()
    print("Mean ONI-RONI gap, 1980-1999:", gap["1980":"1999"].mean().round(2),
          "| 2015-2026:", gap["2015":].mean().round(2), "°C  (≈ background warming signal)")
ax[1].axhline(2.5, ls="--", c="k", lw=1); ax[1].set_title("Raw vs relative Niño-3.4"); ax[1].legend()
plt.tight_layout(); plt.show()

print("DMI (IOD) last 6 months:\n", dmi.dropna().tail(6).round(2))

In [ ]:
# Save tidy index table for the website charts
idx = pd.concat([nino, oni.rename("ONI"), soi.rename("SOI"), dmi.rename("DMI")] +
                ([roni.rename("RONI")] if roni is not None else []) +
                ([wwv.rename("WWV")] if wwv is not None else []), axis=1)["1979":]
idx.to_csv("enso_indices_monthly.csv", index_label="date")
idx.tail()

## Part B — Earth Engine maps
Set your Cloud project ID below. Outputs export to Google Drive folder `le-gis_enso`.

In [ ]:
import ee, geemap
PROJECT = "YOUR-EE-PROJECT"   # <-- change
ee.Authenticate(); ee.Initialize(project=PROJECT)

indo = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Indonesia"))
prov = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq("ADM0_NAME", "Indonesia"))
INDO_BOX = ee.Geometry.BBox(94, -11.5, 141.5, 6.5)
PACIFIC = ee.Geometry.BBox(100, -30, -70, 30)   # crosses the dateline
PAC_CRS = "EPSG:3832"                            # Pacific-centred Mercator
FOLDER = "le-gis_enso"

### Pacific SST anomaly — monthly animation 2026 (OISST v2.1)

In [ ]:
oisst = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("anom")
last_sst = ee.Date(oisst.aggregate_max("system:time_start"))
print("OISST latest:", last_sst.format("YYYY-MM-dd").getInfo())

def month_anom(m):
    s = ee.Date.fromYMD(2026, m, 1)
    return (oisst.filterDate(s, s.advance(1, "month")).mean().multiply(0.01)
            .set("system:time_start", s.millis(), "label", s.format("MMM YYYY")))
n_months = last_sst.get("month").getInfo()
sst_months = ee.ImageCollection([month_anom(m) for m in range(1, n_months + 1)])

SST_VIS = {"min": -4, "max": 4, "palette": ["053061","2166ac","4393c3","92c5de","f7f7f7",
                                             "f4a582","d6604d","b2182b","67001f"]}
video = sst_months.map(lambda i: i.visualize(**SST_VIS))
print("Animated GIF:", video.getVideoThumbURL({"region": PACIFIC, "crs": PAC_CRS,
      "dimensions": 900, "framesPerSecond": 1.5}))

latest_sst = sst_months.sort("system:time_start", False).first()
geemap.ee_export_image_to_drive(latest_sst.toFloat(), description="pacific_sst_anom_latest",
      folder=FOLDER, region=PACIFIC, crs=PAC_CRS, scale=27830, maxPixels=1e10)

### Indonesia impact — 2026 dry-season rainfall anomaly (CHIRPS)
Window: 1 June 2026 → latest CHIRPS pentad. Baseline: same window, 1991–2020. Analogues: 2015 and 2023.
Anomaly is masked where baseline rainfall < 50 mm (percent changes are meaningless in near-zero rain).

In [ ]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").select("precipitation")
last = ee.Date(chirps.aggregate_max("system:time_start")).advance(5, "day")
end_str = last.format("YYYY-MM-dd").getInfo()
NDAYS = last.difference(ee.Date("2026-06-01"), "day").getInfo()
print(f"Window: 2026-06-01 -> {end_str} ({NDAYS:.0f} days)")

def window_total(year):
    s = ee.Date.fromYMD(year, 6, 1)
    return chirps.filterDate(s, s.advance(NDAYS, "day")).sum()

clim = ee.ImageCollection([window_total(y) for y in range(1991, 2021)]).mean()
def anom_pct(year):
    return (window_total(year).subtract(clim).divide(clim).multiply(100)
            .updateMask(clim.gte(50)).rename(f"anom_{year}").clip(indo))

a26, a15, a23 = anom_pct(2026), anom_pct(2015), anom_pct(2023)

RAIN_VIS = {"min": -80, "max": 80, "palette": ["67001f","b2182b","f4a582","f7f7f7",
                                               "92c5de","2166ac","053061"]}
Map = geemap.Map(center=(-2.5, 118), zoom=5)
Map.addLayer(a15, RAIN_VIS, "2015 analogue", False)
Map.addLayer(a23, RAIN_VIS, "2023 analogue", False)
Map.addLayer(a26, RAIN_VIS, "2026 rainfall anomaly (%)")
Map.addLayer(prov.style(color="333333", fillColor="00000000", width=0.6), {}, "Provinces")
Map.add_colorbar(RAIN_VIS, label="Rainfall anomaly vs 1991–2020 (%)")
Map

In [ ]:
# Province ranking — the choropleth for the website
stats = a26.addBands(a15).addBands(a23).reduceRegions(prov, ee.Reducer.mean(), scale=5566)
pdf = geemap.ee_to_df(stats)[["ADM1_NAME", "anom_2026", "anom_2015", "anom_2023"]]
pdf = pdf.sort_values("anom_2026").round(1)
pdf.to_csv("indonesia_province_rain_anom_2026.csv", index=False)
display(pdf.head(15))

geemap.ee_export_image_to_drive(a26.toFloat(), description="indo_rain_anom_2026",
      folder=FOLDER, region=INDO_BOX, scale=5566, crs="EPSG:4326", maxPixels=1e10)
ee.batch.Export.table.toDrive(stats.select(["ADM1_NAME","anom_2026","anom_2015","anom_2023"]),
      description="indo_province_rain_anom", folder=FOLDER, fileFormat="GeoJSON").start()
print("Exports started — check https://code.earthengine.google.com/tasks")

### Teleconnection — Indonesia monthly rainfall vs Niño-3.4 (and IOD), 2000–2026

In [ ]:
clim_m = {m: chirps.filter(ee.Filter.calendarRange(m, m, "month"))
                   .filterDate("1991-01-01", "2021-01-01").sum().divide(30) for m in range(1, 13)}
months = pd.date_range("2000-01-01", end_str, freq="MS")[:-1]   # complete months only

def month_feat(t):
    s = ee.Date(t.strftime("%Y-%m-%d"))
    tot = chirps.filterDate(s, s.advance(1, "month")).sum()
    a = tot.subtract(clim_m[t.month]).divide(clim_m[t.month].max(1)).multiply(100)
    v = a.reduceRegion(ee.Reducer.mean(), indo.geometry(), scale=25000, maxPixels=1e10)
    return ee.Feature(None, {"date": t.strftime("%Y-%m"), "anom": v.get("precipitation")})

ts = ee.FeatureCollection([month_feat(t) for t in months]).getInfo()
rain = pd.Series({pd.Timestamp(f["properties"]["date"]): f["properties"]["anom"] for f in ts["features"]})

j = pd.concat([rain.rename("rain"), nino["Niño-3.4"], dmi.rename("DMI")], axis=1).dropna()
dry = j[j.index.month.isin([7, 8, 9, 10])]   # dry season, when ENSO/IOD bite hardest
print("Jul–Oct correlation with Indonesia rainfall anomaly:\n", dry.corr()["rain"].round(2))

fig, ax1 = plt.subplots(figsize=(14, 4.5))
ax1.bar(rain.index, rain, width=25, color=np.where(rain < 0, "#b2182b", "#2166ac"))
ax1.set_ylabel("Indonesia rainfall anomaly (%)")
ax2 = ax1.twinx(); ax2.plot(nino.index[nino.index >= "2000"], nino["Niño-3.4"]["2000":], c="k", lw=1.5)
ax2.set_ylabel("Niño-3.4 (°C)"); ax1.set_title("Indonesia rainfall vs Niño-3.4")
plt.show()
rain.to_csv("indonesia_monthly_rain_anom.csv", header=["anom_pct"])

## Outputs for le-gis.com
- `enso_indices_monthly.csv` — index charts (event comparison, ONI vs RONI, WWV lead)
- `indonesia_province_rain_anom_2026.csv` + Drive `indo_province_rain_anom.geojson` — province choropleth
- Drive `indo_rain_anom_2026.tif`, `pacific_sst_anom_latest.tif` — COG/tiles for the web map
- Pacific SST GIF URL — story header animation

**Next — Notebook 02:** ERA5 zonal wind + GODAS/ORAS5 20°C-isotherm Hovmöllers (H2 trigger, Kelvin waves).

**References:** Bjerknes 1969 *MWR*; Wyrtki 1975 *JPO*; Jin 1997 *JAS*; Harrison & Vecchi 1997 *J. Climate*; Saji et al. 1999 *Nature*;
McPhaden et al. 2006 *Science*; Cai et al. 2014 *Nat. Clim. Change*; Timmermann et al. 2018 *Nature*; van Oldenborgh et al. 2021 *ERL*;
NOAA CPC ENSO Diagnostic Discussion, 8 Oct 2026. Data: NOAA CPC/PSL/PMEL, NOAA OISST v2.1, CHIRPS (Funk et al. 2015), FAO GAUL 2015.